# RQ1 paired Z-test

Original notebook used to test paired reconstruction errors from model-generated `pred_test.csv` files.


## Path configuration

The analysis logic below is unchanged from the research notebook. In the
configuration cell, replace every `REPLACE_WITH_...` value with the path to
the named file or directory on your machine. Paths may be absolute, or
relative to the repository root when Jupyter is started from that directory.
Generated outputs are written under the repository's `outputs/` directory.


# RQ1 Paired One-Tailed Z-Test: G-NCDM-poly as Reference










|---|---|---|





\[
AE_{model,j}=|score_j-pred_{model,j}|
\]

\[
AE_{reference,j}=|score_j-pred_{reference,j}|
\]


\[
d_j = AE_candidate,j - AE_G-NCDM-poly,j
\]


\[
H_0: \mu_d \ge 0, \qquad H_1: \mu_d < 0
\]

\[
z = \frac{\bar d}{s_d/\sqrt n}, \qquad p_{raw}=\Phi(z)
\]


1. `p_value_raw < 0.05`
2. `mean_paired_difference < 0`
3. `mae_model < mae_reference`






In [ ]:
# CELL A1 — imports, exact source paths, constants, expected MAE
"""Validated observation-level paired one-tailed z-tests for RQ1.

Every comparison pairs rows by (user_id, item_id), calculates one absolute-error
difference per observation, and returns one p-value per model x setting. If any
required validation fails, the whole run stops before significance is computed.
"""

from __future__ import annotations

import argparse

import math

import sys

from dataclasses import dataclass

from pathlib import Path

from typing import Iterable

import numpy as np

import pandas as pd

REQUIRED_COLUMNS = ("user_id", "item_id", "score", "pred")

SETTINGS = ("S1", "S2", "S3")

ALPHA = 0.05

MODEL_FILES = {
    "gncdm_poly": "gncdm_poly.csv",
    "gllmcdm": "gllmcdm.csv",
    "ncdm_poly": "ncdm_poly.csv",
    "gllmcdm_wo_attempt": "gllmcdm_wo_attempt.csv",
    "llm_only": "llm_only.csv",
}

MODEL_NAMES = {
    "gncdm_poly": "G-NCDM-poly",
    "gllmcdm": "GLLMCDM",
    "ncdm_poly": "NCDM-poly",
    "gllmcdm_wo_attempt": "GLLMCDM w/o attempt",
    "llm_only": "LLM-only",
}

EXPECTED_MAE = {
    ("S1", "gncdm_poly"): 1.2782,
    ("S2", "gncdm_poly"): 1.2806,
    ("S3", "gncdm_poly"): 3.4756,
    ("S1", "gllmcdm"): 1.5212,
    ("S2", "gllmcdm"): 1.6333,
    ("S3", "gllmcdm"): 1.4971,
    ("S1", "ncdm_poly"): 1.8651,
    ("S2", "ncdm_poly"): 2.5547,
    ("S3", "ncdm_poly"): 3.9392,
    ("S1", "gllmcdm_wo_attempt"): 1.8846,
    ("S2", "gllmcdm_wo_attempt"): 2.4855,
    ("S3", "gllmcdm_wo_attempt"): 2.0235,
}

VALIDATION_COLUMNS = [
    "setting",
    "model",
    "reference_model",
    "scope",
    "check",
    "status",
    "observed",
    "expected",
    "tolerance",
    "details",
    "path",
]

PAIRED_COLUMNS = [
    "setting",
    "model",
    "reference_model",
    "user_id",
    "item_id",
    "score",
    "pred_model",
    "pred_reference",
    "ae_model",
    "ae_reference",
    "paired_difference",
]

RESULT_COLUMNS = [
    "setting",
    "model",
    "reference_model",
    "n_pairs",
    "mae_model",
    "mae_reference",
    "mae_difference",
    "mean_paired_difference",
    "sd_paired_difference",
    "standard_error",
    "z_statistic",
    "alternative",
    "p_value_raw",
    "significant_raw_0_05",
    "p_value_holm",
    "significant_holm_0_05",
    "conclusion",
]

@dataclass(frozen=True)
class Comparison:
    setting: str
    model: str
    reference: str


# Explicit saved-prediction paths supplied for this RQ1 analysis.
# Paths are relative to PROJECT_ROOT; B2 displays the resolved absolute paths.
SOURCE_PATHS = {
    ("S1", "gllmcdm"): Path("REPLACE_WITH_GLLMCDM_S1_PRED_TEST_CSV"),
    ("S1", "gllmcdm_wo_attempt"): Path("REPLACE_WITH_GLLMCDM_WO_ATTEMPT_S1_PRED_TEST_CSV"),
    ("S1", "gncdm_poly"): Path("REPLACE_WITH_GNCDM_POLY_S1_PRED_TEST_CSV"),
    ("S1", "ncdm_poly"): Path("REPLACE_WITH_NCDM_POLY_S1_PRED_TEST_CSV"),
    ("S1", "llm_only"): Path("REPLACE_WITH_LLM_ONLY_S1_PRED_TEST_CSV"),
    ("S2", "gllmcdm"): Path("REPLACE_WITH_GLLMCDM_S2_PRED_TEST_CSV"),
    ("S2", "gllmcdm_wo_attempt"): Path("REPLACE_WITH_GLLMCDM_WO_ATTEMPT_S2_PRED_TEST_CSV"),
    ("S2", "gncdm_poly"): Path("REPLACE_WITH_GNCDM_POLY_S2_PRED_TEST_CSV"),
    ("S2", "ncdm_poly"): Path("REPLACE_WITH_NCDM_POLY_S2_PRED_TEST_CSV"),
    ("S2", "llm_only"): Path("REPLACE_WITH_LLM_ONLY_S2_PRED_TEST_CSV"),
    ("S3", "gllmcdm"): Path("REPLACE_WITH_GLLMCDM_S3_PRED_TEST_CSV"),
    ("S3", "gllmcdm_wo_attempt"): Path("REPLACE_WITH_GLLMCDM_WO_ATTEMPT_S3_PRED_TEST_CSV"),
    ("S3", "gncdm_poly"): Path("REPLACE_WITH_GNCDM_POLY_S3_PRED_TEST_CSV"),
    ("S3", "ncdm_poly"): Path("REPLACE_WITH_NCDM_POLY_S3_PRED_TEST_CSV"),
    ("S3", "llm_only"): Path("REPLACE_WITH_LLM_ONLY_S3_PRED_TEST_CSV"),
}


In [ ]:
# CELL A2 — comparison definitions and exact prediction-path resolver
def add_validation(
    rows: list[dict],
    *,
    setting: str,
    model: str,
    reference: str = "",
    scope: str,
    check: str,
    status: str,
    observed: object = "",
    expected: object = "",
    tolerance: object = "",
    details: str = "",
    path: Path | str = "",
) -> None:
    rows.append(
        {
            "setting": setting,
            "model": MODEL_NAMES.get(model, model),
            "reference_model": MODEL_NAMES.get(reference, reference) if reference else "",
            "scope": scope,
            "check": check,
            "status": status,
            "observed": observed,
            "expected": expected,
            "tolerance": tolerance,
            "details": details,
            "path": str(path),
        }
    )


def parse_include(raw: str | None, allowed: Iterable[str]) -> list[str]:
    allowed_list = list(allowed)
    if raw is None:
        return allowed_list
    selected = [part.strip() for part in raw.split(",") if part.strip()]
    invalid = sorted(set(selected) - set(allowed_list))
    if invalid:
        raise ValueError(
            f"Unknown --include value(s): {', '.join(invalid)}. "
            f"Allowed: {', '.join(allowed_list)}"
        )
    if not selected:
        raise ValueError("--include must contain at least one model slug")
    return selected


def build_comparisons(mode: str, include: str | None) -> list[Comparison]:
    if mode == "vs_gncdm":
        candidates = parse_include(
            include, ("gllmcdm", "ncdm_poly", "gllmcdm_wo_attempt", "llm_only")
        )
        comparisons = []
        for setting in SETTINGS:
            for candidate in candidates:
                comparisons.append(Comparison(setting, candidate, "gncdm_poly"))
        return comparisons

    if mode == "vs_gllmcdm":
        competitors = parse_include(
            include, ("gncdm_poly", "ncdm_poly", "gllmcdm_wo_attempt", "llm_only")
        )
        comparisons = []
        for setting in SETTINGS:
            for competitor in competitors:
                # GLLMCDM stays on the left: d = AE_GLLMCDM - AE_competitor.
                comparisons.append(Comparison(setting, "gllmcdm", competitor))
        return comparisons

    raise ValueError(f"Unsupported mode: {mode}")


def prediction_path(input_root: Path, setting: str, model: str) -> Path:
    """Resolve one exact saved-prediction path; never guess a similar filename."""
    key = (setting, model)
    if key not in SOURCE_PATHS:
        raise KeyError(f"No saved-prediction path configured for {key}")
    return Path(SOURCE_PATHS[key])


In [ ]:
# CELL A3 — validate and load one prediction CSV
def canonicalize_gncdm_open_world_keys(
    df: pd.DataFrame,
    path: Path,
    setting: str,
    model: str,
) -> tuple[pd.DataFrame, str]:
    """Restore original G-NCDM S2/S3 keys when reading raw model outputs.

    G-NCDM uses local/internal coordinates for open-world evaluation. Staged
    files that already contain source_user_id/source_item_id are already
    canonical and are left unchanged. Raw S2 mappings are inverted from the
    deterministic sorted-ID maps used during inference. Raw S3 files retain
    original_user_id/original_item_id directly. Every recovered key and score
    is checked against the corresponding source test split without using row
    order.
    """
    if model != "gncdm_poly" or setting not in {"S2", "S3"}:
        return df, "not required"
    if {"source_user_id", "source_item_id"} <= set(df.columns):
        return df, "already canonicalized in staged prediction file"

    gncdm_root = path.parents[2]
    split_dir = gncdm_root / "data" / "Exp_data" / f"{setting}_split"
    test_path = split_dir / "test.csv"
    if not test_path.is_file():
        raise ValueError(f"Cannot canonicalize {setting}; missing {test_path}")
    test = pd.read_csv(test_path)
    transformed = df.copy()

    if setting == "S2":
        train_path = split_dir / "train.csv"
        if not train_path.is_file():
            raise ValueError(f"Cannot canonicalize S2; missing {train_path}")
        train = pd.read_csv(train_path)
        original_users = sorted(test["user_id"].astype(int).unique().tolist())
        original_items = sorted(train["item_id"].astype(int).unique().tolist())
        inverse_user_map = dict(enumerate(original_users))
        inverse_item_map = dict(enumerate(original_items))
        transformed["user_id"] = (
            transformed["user_id"].astype(int).map(inverse_user_map)
        )
        transformed["item_id"] = (
            transformed["item_id"].astype(int).map(inverse_item_map)
        )
        method = "restored from deterministic S2 sorted-ID maps"
    else:
        needed = {"original_user_id", "original_item_id"}
        missing = needed - set(transformed.columns)
        if missing:
            raise ValueError(
                f"Cannot canonicalize S3; missing columns {sorted(missing)}"
            )
        transformed["user_id"] = transformed["original_user_id"]
        transformed["item_id"] = transformed["original_item_id"]
        method = "restored from original_user_id/original_item_id"

    if transformed[["user_id", "item_id"]].isna().any().any():
        raise ValueError(f"{setting} contains an internal ID with no inverse mapping")
    transformed["user_id"] = transformed["user_id"].astype(int)
    transformed["item_id"] = transformed["item_id"].astype(int)

    audit = transformed[["user_id", "item_id", "score"]].merge(
        test[["user_id", "item_id", "score"]],
        on=["user_id", "item_id"],
        how="outer",
        suffixes=("_prediction", "_test"),
        indicator=True,
        validate="one_to_one",
    )
    if not (audit["_merge"] == "both").all():
        counts = audit["_merge"].value_counts().to_dict()
        raise ValueError(f"Recovered {setting} keys do not match test.csv: {counts}")
    score_difference = (audit["score_prediction"] - audit["score_test"]).abs()
    if not np.isfinite(score_difference).all() or float(score_difference.max()) > 1e-12:
        raise ValueError(
            f"Recovered {setting} scores differ from test.csv; "
            f"max difference={float(score_difference.max())}"
        )
    # validate_and_load_file reads every observation key as pandas StringDtype.
    # Restore that common dtype after the integer-based source-split audit so
    # downstream key merges compare like with like.
    transformed["user_id"] = transformed["user_id"].astype("string")
    transformed["item_id"] = transformed["item_id"].astype("string")
    return transformed, method


def validate_and_load_file(
    path: Path,
    setting: str,
    model: str,
    mae_tolerance: float,
    rows: list[dict],
) -> pd.DataFrame | None:
    display = MODEL_NAMES[model]
    if not path.is_file():
        add_validation(
            rows,
            setting=setting,
            model=model,
            scope="file",
            check="file_exists",
            status="FAIL",
            expected="existing CSV",
            details=f"Missing prediction file for {display}",
            path=path,
        )
        return None

    add_validation(
        rows,
        setting=setting,
        model=model,
        scope="file",
        check="file_exists",
        status="PASS",
        observed="existing CSV",
        path=path,
    )

    try:
        df = pd.read_csv(
            path,
            dtype={"user_id": "string", "item_id": "string"},
            keep_default_na=False,
        )
    except Exception as exc:  # report parse failures in the requested CSV report
        add_validation(
            rows,
            setting=setting,
            model=model,
            scope="file",
            check="csv_readable",
            status="FAIL",
            details=f"{type(exc).__name__}: {exc}",
            path=path,
        )
        return None

    missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    add_validation(
        rows,
        setting=setting,
        model=model,
        scope="file",
        check="required_columns",
        status="FAIL" if missing_columns else "PASS",
        observed=",".join(map(str, df.columns)),
        expected=",".join(REQUIRED_COLUMNS),
        details=(f"Missing columns: {', '.join(missing_columns)}" if missing_columns else ""),
        path=path,
    )
    if missing_columns:
        return None

    try:
        df, canonicalization_method = canonicalize_gncdm_open_world_keys(
            df, path, setting, model
        )
        add_validation(
            rows,
            setting=setting,
            model=model,
            scope="file",
            check="canonical_observation_keys",
            status="PASS",
            observed=canonicalization_method,
            expected="original observation coordinates",
            details="No row-order matching is used",
            path=path,
        )
    except Exception as exc:
        add_validation(
            rows,
            setting=setting,
            model=model,
            scope="file",
            check="canonical_observation_keys",
            status="FAIL",
            expected="original observation coordinates",
            details=f"{type(exc).__name__}: {exc}",
            path=path,
        )
        return None

    df = df.loc[:, REQUIRED_COLUMNS].copy()
    blank_key = (
        df["user_id"].astype("string").str.strip().eq("")
        | df["item_id"].astype("string").str.strip().eq("")
    )
    duplicate_key = df.duplicated(["user_id", "item_id"], keep=False)
    add_validation(
        rows,
        setting=setting,
        model=model,
        scope="file",
        check="observation_keys_present",
        status="FAIL" if blank_key.any() else "PASS",
        observed=int(blank_key.sum()),
        expected=0,
        details="blank user_id/item_id rows",
        path=path,
    )
    add_validation(
        rows,
        setting=setting,
        model=model,
        scope="file",
        check="duplicate_observation_keys",
        status="FAIL" if duplicate_key.any() else "PASS",
        observed=int(duplicate_key.sum()),
        expected=0,
        details="rows belonging to duplicated (user_id, item_id) keys",
        path=path,
    )

    score_numeric = pd.to_numeric(df["score"], errors="coerce")
    pred_numeric = pd.to_numeric(df["pred"], errors="coerce")
    invalid_numeric = ~(np.isfinite(score_numeric) & np.isfinite(pred_numeric))
    add_validation(
        rows,
        setting=setting,
        model=model,
        scope="file",
        check="finite_score_and_prediction",
        status="FAIL" if invalid_numeric.any() else "PASS",
        observed=int(invalid_numeric.sum()),
        expected=0,
        details="rows with missing, non-numeric, or non-finite score/pred",
        path=path,
    )

    if blank_key.any() or duplicate_key.any() or invalid_numeric.any():
        return None

    df["score"] = score_numeric.astype(float)
    df["pred"] = pred_numeric.astype(float)
    mae = float((df["score"] - df["pred"]).abs().mean())
    expected_mae = EXPECTED_MAE.get((setting, model))
    if expected_mae is None:
        add_validation(
            rows,
            setting=setting,
            model=model,
            scope="file",
            check="mae_matches_reported_table",
            status="NOT_APPLICABLE",
            observed=mae,
            details="No Table RQ1 value exists for this newly added baseline",
            path=path,
        )
    else:
        difference = abs(mae - expected_mae)
        add_validation(
            rows,
            setting=setting,
            model=model,
            scope="file",
            check="mae_matches_reported_table",
            status="PASS" if difference <= mae_tolerance else "FAIL",
            observed=mae,
            expected=expected_mae,
            tolerance=mae_tolerance,
            details=f"absolute difference={difference:.12g}",
            path=path,
        )
    return df

In [ ]:
# CELL A4 — pair observations by key and calculate differences
def validate_pair(
    comparison: Comparison,
    model_df: pd.DataFrame | None,
    reference_df: pd.DataFrame | None,
    score_tolerance: float,
    rows: list[dict],
) -> pd.DataFrame | None:
    if model_df is None or reference_df is None:
        add_validation(
            rows,
            setting=comparison.setting,
            model=comparison.model,
            reference=comparison.reference,
            scope="pair",
            check="pair_ready",
            status="FAIL",
            details="One or both input files failed file-level validation",
        )
        return None

    model_keys = model_df[["user_id", "item_id"]]
    reference_keys = reference_df[["user_id", "item_id"]]
    key_check = model_keys.merge(
        reference_keys,
        on=["user_id", "item_id"],
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    model_only = int((key_check["_merge"] == "left_only").sum())
    reference_only = int((key_check["_merge"] == "right_only").sum())
    keys_match = model_only == 0 and reference_only == 0
    add_validation(
        rows,
        setting=comparison.setting,
        model=comparison.model,
        reference=comparison.reference,
        scope="pair",
        check="observation_sets_match",
        status="PASS" if keys_match else "FAIL",
        observed=f"model_only={model_only}; reference_only={reference_only}",
        expected="model_only=0; reference_only=0",
        details="Pairing is by (user_id, item_id), never by row order",
    )
    if not keys_match:
        return None

    paired = model_df.rename(
        columns={"score": "score_model", "pred": "pred_model"}
    ).merge(
        reference_df.rename(
            columns={"score": "score_reference", "pred": "pred_reference"}
        ),
        on=["user_id", "item_id"],
        how="inner",
        validate="one_to_one",
        sort=True,
    )
    score_delta = (paired["score_model"] - paired["score_reference"]).abs()
    score_match = bool((score_delta <= score_tolerance).all())
    add_validation(
        rows,
        setting=comparison.setting,
        model=comparison.model,
        reference=comparison.reference,
        scope="pair",
        check="ground_truth_scores_match",
        status="PASS" if score_match else "FAIL",
        observed=float(score_delta.max()) if len(score_delta) else math.nan,
        expected=0.0,
        tolerance=score_tolerance,
        details="maximum absolute score difference after key-based pairing",
    )
    if not score_match:
        return None

    if len(paired) < 2:
        add_validation(
            rows,
            setting=comparison.setting,
            model=comparison.model,
            reference=comparison.reference,
            scope="pair",
            check="minimum_pair_count",
            status="FAIL",
            observed=len(paired),
            expected=">=2",
            details="At least two paired observations are required for sample SD",
        )
        return None

    add_validation(
        rows,
        setting=comparison.setting,
        model=comparison.model,
        reference=comparison.reference,
        scope="pair",
        check="minimum_pair_count",
        status="PASS",
        observed=len(paired),
        expected=">=2",
    )

    paired["score"] = paired["score_model"]
    paired["ae_model"] = (paired["score"] - paired["pred_model"]).abs()
    paired["ae_reference"] = (paired["score"] - paired["pred_reference"]).abs()
    paired["paired_difference"] = paired["ae_model"] - paired["ae_reference"]
    sd = float(paired["paired_difference"].std(ddof=1))
    nonzero_sd = math.isfinite(sd) and sd > 0.0
    add_validation(
        rows,
        setting=comparison.setting,
        model=comparison.model,
        reference=comparison.reference,
        scope="pair",
        check="nonzero_finite_sd_difference",
        status="PASS" if nonzero_sd else "FAIL",
        observed=sd,
        expected=">0 and finite",
        details="A zero/invalid SD makes the requested z statistic undefined",
    )
    if not nonzero_sd:
        return None

    return paired

In [ ]:
# CELL A5 — z-test, raw p-value, Holm adjustment, conclusion
def normal_cdf(z_value: float) -> float:
    # erfc keeps substantially more precision than 1 + erf(z) in the far
    # negative tail, where the one-tailed p-value can otherwise round to zero.
    return 0.5 * math.erfc(-z_value / math.sqrt(2.0))


def holm_adjust(p_values: list[float]) -> list[float]:
    count = len(p_values)
    order = sorted(range(count), key=lambda index: p_values[index])
    adjusted = [math.nan] * count
    running_max = 0.0
    for rank, index in enumerate(order):
        candidate = min(1.0, (count - rank) * p_values[index])
        running_max = max(running_max, candidate)
        adjusted[index] = running_max
    return adjusted


def calculate_result(comparison: Comparison, paired: pd.DataFrame) -> dict:
    differences = paired["paired_difference"].to_numpy(dtype=float)
    n_pairs = int(len(differences))
    mean_difference = float(np.mean(differences))
    sd_difference = float(np.std(differences, ddof=1))
    standard_error = sd_difference / math.sqrt(n_pairs)
    z_statistic = mean_difference / standard_error
    p_value = normal_cdf(z_statistic)
    mae_model = float(paired["ae_model"].mean())
    mae_reference = float(paired["ae_reference"].mean())
    significant = p_value < ALPHA
    all_better_conditions = significant and mean_difference < 0 and mae_model < mae_reference

    if all_better_conditions:
        conclusion = (
            f"{MODEL_NAMES[comparison.model]} has significantly lower absolute error "
            f"than {MODEL_NAMES[comparison.reference]} (raw one-tailed p<0.05)."
        )
    elif mae_model >= mae_reference or mean_difference >= 0:
        conclusion = (
            f"No evidence that {MODEL_NAMES[comparison.model]} has lower absolute error "
            f"than {MODEL_NAMES[comparison.reference]}; its paired mean difference is not negative."
        )
    else:
        conclusion = (
            f"{MODEL_NAMES[comparison.model]} has lower MAE, but the difference is not "
            f"statistically significant at raw alpha=0.05."
        )

    return {
        "setting": comparison.setting,
        "model": MODEL_NAMES[comparison.model],
        "reference_model": MODEL_NAMES[comparison.reference],
        "n_pairs": n_pairs,
        "mae_model": mae_model,
        "mae_reference": mae_reference,
        "mae_difference": mae_model - mae_reference,
        "mean_paired_difference": mean_difference,
        "sd_paired_difference": sd_difference,
        "standard_error": standard_error,
        "z_statistic": z_statistic,
        "alternative": "less",
        "p_value_raw": p_value,
        "significant_raw_0_05": all_better_conditions,
        "p_value_holm": math.nan,
        "significant_holm_0_05": False,
        "conclusion": conclusion,
    }

In [ ]:
# CELL A6 — format paired data and write CSV/TXT outputs
def paired_output_frame(comparison: Comparison, paired: pd.DataFrame) -> pd.DataFrame:
    output = paired.loc[
        :,
        [
            "user_id",
            "item_id",
            "score",
            "pred_model",
            "pred_reference",
            "ae_model",
            "ae_reference",
            "paired_difference",
        ],
    ].copy()
    output.insert(0, "reference_model", MODEL_NAMES[comparison.reference])
    output.insert(0, "model", MODEL_NAMES[comparison.model])
    output.insert(0, "setting", comparison.setting)
    return output.loc[:, PAIRED_COLUMNS]


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    frame.to_csv(path, index=False, float_format="%.12g")


def write_summary(
    path: Path,
    mode: str,
    mae_tolerance: float,
    score_tolerance: float,
    validation_failed: bool,
    validation_rows: list[dict],
    results: pd.DataFrame,
) -> None:
    if mode == "vs_gncdm":
        question = "Does each candidate model have lower absolute error than G-NCDM-poly?"
        direction = "d = AE_candidate - AE_G-NCDM-poly; H1: mean(d) < 0"
    else:
        question = "Does GLLMCDM have lower absolute error than each competitor?"
        direction = "d = AE_GLLMCDM - AE_competitor; H1: mean(d) < 0"

    lines = [
        "RQ1 paired one-tailed z-test summary",
        "====================================",
        f"Question: {question}",
        f"Direction: {direction}",
        "Pairing key: (user_id, item_id); observations are never paired by row order.",
        "Settings: S1, S2, and S3 are tested separately; observations are never pooled.",
        "Test: z = mean(d) / (sample_sd(d) / sqrt(n)); p_raw = NormalCDF(z).",
        f"Raw alpha: {ALPHA}",
        f"Reported-MAE tolerance: {mae_tolerance}",
        f"Ground-truth score tolerance: {score_tolerance}",
        "Holm-adjusted p-values are additional columns and do not replace raw p-values.",
        "",
    ]
    if validation_failed:
        failed = [row for row in validation_rows if row["status"] == "FAIL"]
        lines.extend(
            [
                "STATUS: ABORTED BEFORE SIGNIFICANCE TESTING",
                f"Failed validation checks: {len(failed)}",
                "No z statistics or p-values were calculated.",
                "",
                "Failures:",
            ]
        )
        for row in failed:
            pair = row["model"]
            if row["reference_model"]:
                pair += f" vs {row['reference_model']}"
            lines.append(
                f"- {row['setting']} | {pair} | {row['check']}: {row['details']} "
                f"(observed={row['observed']}, expected={row['expected']})"
            )
    else:
        lines.extend(["STATUS: COMPLETED", f"Comparisons tested: {len(results)}", "", "Results:"])
        for _, row in results.iterrows():
            lines.append(
                f"- {row['setting']} | {row['model']} vs {row['reference_model']} | "
                f"n={int(row['n_pairs'])}, MAE={row['mae_model']:.6f} vs "
                f"{row['mae_reference']:.6f}, mean(d)={row['mean_paired_difference']:.6f}, "
                f"z={row['z_statistic']:.6f}, p_raw={row['p_value_raw']:.12g}, "
                f"p_Holm={row['p_value_holm']:.12g} | {row['conclusion']}"
            )
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")

In [ ]:
# CELL A7 — fail-fast analysis pipeline
def run_analysis(
    *,
    mode: str,
    predictions_dir: Path,
    output_dir: Path,
    include: str | None,
    mae_tolerance: float,
    score_tolerance: float,
) -> int:
    comparisons = build_comparisons(mode, include)
    output_dir.mkdir(parents=True, exist_ok=True)

    validation_rows: list[dict] = []
    file_cache: dict[tuple[str, str], pd.DataFrame | None] = {}
    needed_files = sorted(
        {(c.setting, c.model) for c in comparisons}
        | {(c.setting, c.reference) for c in comparisons}
    )
    for setting, model in needed_files:
        path = prediction_path(predictions_dir, setting, model)
        file_cache[(setting, model)] = validate_and_load_file(
            path, setting, model, mae_tolerance, validation_rows
        )

    paired_by_comparison: list[tuple[Comparison, pd.DataFrame | None]] = []
    for comparison in comparisons:
        paired = validate_pair(
            comparison,
            file_cache.get((comparison.setting, comparison.model)),
            file_cache.get((comparison.setting, comparison.reference)),
            score_tolerance,
            validation_rows,
        )
        paired_by_comparison.append((comparison, paired))

    validation_frame = pd.DataFrame(validation_rows, columns=VALIDATION_COLUMNS)
    write_csv(validation_frame, output_dir / "validation_report.csv")
    validation_failed = bool((validation_frame["status"] == "FAIL").any())

    if validation_failed:
        write_csv(pd.DataFrame(columns=PAIRED_COLUMNS), output_dir / "paired_observations.csv")
        empty_results = pd.DataFrame(columns=RESULT_COLUMNS)
        write_csv(empty_results, output_dir / "rq1_ztest_results.csv")
        write_summary(
            output_dir / "rq1_summary.txt",
            mode,
            mae_tolerance,
            score_tolerance,
            True,
            validation_rows,
            empty_results,
        )
        print(f"ABORTED: validation failed. See {output_dir / 'validation_report.csv'}")
        return 2

    paired_frames = []
    result_rows = []
    for comparison, paired in paired_by_comparison:
        assert paired is not None
        paired_frames.append(paired_output_frame(comparison, paired))
        result_rows.append(calculate_result(comparison, paired))

    results = pd.DataFrame(result_rows, columns=RESULT_COLUMNS)
    adjusted = holm_adjust(results["p_value_raw"].astype(float).tolist())
    results["p_value_holm"] = adjusted
    results["significant_holm_0_05"] = (
        (results["p_value_holm"] < ALPHA)
        & (results["mean_paired_difference"] < 0)
        & (results["mae_model"] < results["mae_reference"])
    )
    paired_output = pd.concat(paired_frames, ignore_index=True)
    write_csv(paired_output, output_dir / "paired_observations.csv")
    write_csv(results, output_dir / "rq1_ztest_results.csv")
    write_summary(
        output_dir / "rq1_summary.txt",
        mode,
        mae_tolerance,
        score_tolerance,
        False,
        validation_rows,
        results,
    )
    print(f"Completed {len(results)} comparisons. Results: {output_dir}")
    return 0

In [ ]:
MODE = "vs_gncdm"
OUTPUT_DIR_NAME = "rq1_vs_gncdm"

from pathlib import Path
import math

import numpy as np
import pandas as pd
from IPython.display import display

# USER CONFIGURATION
# Prediction CSV paths are configured directly in SOURCE_PATHS (Cell A1).
PROJECT_ROOT = Path.cwd().resolve()
INPUT_ROOT = PROJECT_ROOT  # Retained for the original function interface.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "RQ1_Paired_ZTest" / OUTPUT_DIR_NAME
MAE_TOLERANCE = 0.0001
SCORE_TOLERANCE = 1e-12
INCLUDE = None

print("Project root:", PROJECT_ROOT)
print("Input mode: direct saved pred_test.csv paths from CELL A1")
print("Outputs:", OUTPUT_DIR)
print("Mode:", MODE)

In [ ]:
comparisons = build_comparisons(MODE, INCLUDE)
required_inputs = sorted(
    {(c.setting, c.model) for c in comparisons}
    | {(c.setting, c.reference) for c in comparisons}
)
input_inventory = pd.DataFrame(
    [
        {
            "setting": setting,
            "model": MODEL_NAMES[model],
            "path": str(prediction_path(INPUT_ROOT, setting, model)),
            "exists": prediction_path(INPUT_ROOT, setting, model).is_file(),
        }
        for setting, model in required_inputs
    ]
)
display(input_inventory)
if not input_inventory["exists"].all():
    raise FileNotFoundError("One or more exact saved-prediction paths in CELL A1 are missing.")

In [ ]:
exit_code = run_analysis(
    mode=MODE,
    predictions_dir=INPUT_ROOT,  # run_analysis resolves exact SOURCE_PATHS via prediction_path()
    output_dir=OUTPUT_DIR,
    include=INCLUDE,
    mae_tolerance=MAE_TOLERANCE,
    score_tolerance=SCORE_TOLERANCE,
)
if exit_code != 0:
    raise RuntimeError(
        f"Analysis stopped with exit code {exit_code}. "
        f"Read {OUTPUT_DIR / 'validation_report.csv'} before proceeding."
    )

In [ ]:
validation = pd.read_csv(OUTPUT_DIR / "validation_report.csv")
status_summary = (
    validation.groupby(["status"], dropna=False)
    .size()
    .rename("n_checks")
    .reset_index()
)
display(status_summary)

failed = validation.loc[validation["status"].eq("FAIL")]
if not failed.empty:
    display(failed)
    raise RuntimeError("Validation contains FAIL; statistical results must not be interpreted.")

print("Validation passed: no FAIL rows. NOT_APPLICABLE is allowed only when no paper MAE exists.")

In [ ]:
mae_reconciliation = validation.loc[
    validation["check"].eq("mae_matches_reported_table"),
    ["setting", "model", "status", "observed", "expected", "tolerance", "details"],
].sort_values(["setting", "model"])
display(mae_reconciliation)

In [ ]:
results = pd.read_csv(OUTPUT_DIR / "rq1_ztest_results.csv")
result_view = results[
    [
        "setting", "model", "reference_model", "n_pairs",
        "mae_model", "mae_reference", "mae_difference",
        "mean_paired_difference", "sd_paired_difference", "standard_error",
        "z_statistic", "alternative", "p_value_raw", "significant_raw_0_05",
        "p_value_holm", "significant_holm_0_05", "conclusion",
    ]
].copy()
display(result_view)

In [ ]:
paired = pd.read_csv(OUTPUT_DIR / "paired_observations.csv")
verification_rows = []

for _, result in results.iterrows():
    subset = paired.loc[
        paired["setting"].eq(result["setting"])
        & paired["model"].eq(result["model"])
        & paired["reference_model"].eq(result["reference_model"])
    ]
    d = subset["paired_difference"].to_numpy(dtype=float)
    mean_d = float(np.mean(d))
    sd_d = float(np.std(d, ddof=1))
    se = sd_d / math.sqrt(len(d))
    z = mean_d / se
    p = 0.5 * math.erfc(-z / math.sqrt(2.0))
    matches_saved_result = all(
        [
            len(d) == int(result["n_pairs"]),
            np.isclose(mean_d, result["mean_paired_difference"], rtol=1e-10, atol=1e-10),
            np.isclose(sd_d, result["sd_paired_difference"], rtol=1e-10, atol=1e-10),
            np.isclose(se, result["standard_error"], rtol=1e-10, atol=1e-10),
            np.isclose(z, result["z_statistic"], rtol=1e-10, atol=1e-10),
            np.isclose(p, result["p_value_raw"], rtol=1e-9, atol=1e-16),
        ]
    )
    verification_rows.append(
        {
            "setting": result["setting"],
            "model": result["model"],
            "reference_model": result["reference_model"],
            "n_recomputed": len(d),
            "mean_d_recomputed": mean_d,
            "z_recomputed": z,
            "p_raw_recomputed": p,
            "matches_saved_result": matches_saved_result,
        }
    )

verification = pd.DataFrame(verification_rows)
display(verification)
assert verification["matches_saved_result"].all()
print("Independent recomputation from paired_observations.csv matches every saved result.")

In [ ]:
print(f"paired_observations.csv contains {len(paired):,} rows across {len(results)} comparisons.")
display(paired.head(12))

In [ ]:
raw_significant = results.loc[
    results["significant_raw_0_05"],
    ["setting", "model", "reference_model", "p_value_raw", "conclusion"],
]
holm_significant = results.loc[
    results["significant_holm_0_05"],
    ["setting", "model", "reference_model", "p_value_holm"],
]

print("Comparisons meeting all raw significance conditions:")
display(raw_significant if not raw_significant.empty else pd.DataFrame({"result": ["None"]}))
print("Comparisons remaining significant after Holm adjustment:")
display(holm_significant if not holm_significant.empty else pd.DataFrame({"result": ["None"]}))

In [ ]:
output_files = [
    OUTPUT_DIR / "validation_report.csv",
    OUTPUT_DIR / "paired_observations.csv",
    OUTPUT_DIR / "rq1_ztest_results.csv",
    OUTPUT_DIR / "rq1_summary.txt",
]
display(pd.DataFrame({"output_file": [str(path) for path in output_files], "exists": [path.is_file() for path in output_files]}))